# Target 3 — Warehouse Management
## Warehouse Fill Forecast + Routing Recommendation (v3 — expanded feature engineering)

**Goal:** Predict how full each warehouse will be N weeks ahead, and recommend where to route new orders.

**Why v3 exists:** in v2, test R2 was stuck around **0.10** even after fixing the trivial "copy current value forward" leakage problem. The delta-target fix was correct, but the feature set was too thin (9 raw features, no memory of recent history) for the model to find real signal. v3 keeps every leakage-safe fix from v2 and adds a much richer, still-leakage-safe feature set: lag features, rolling statistics, momentum/trend features, cyclical calendar encoding, capacity-normalized flow ratios, and a region-relative fill feature.

| File | Columns | Role |
|---|---|---|
| warehouse.parquet | capacity, current_load, inbound_orders, outbound_orders | Current snapshot / net flow |
| inventory.parquet | stock_level, replenishment_date | Real inbound signal (per warehouse, per week) |
| orders.parquet | region, created_at | Real outbound/demand signal (per region, per week) |
| holidays.parquet | date, event_name | Demand-spike feature |
| stores.parquet | latitude, longitude, region | Warehouse location proxy (region centroid) |

**Data-leakage discipline used throughout this notebook:**
- The train/test split is **chronological** (train = earlier weeks, test = the most recent ~15% of weeks) — never a random split, because this is a forecasting problem.
- The prediction target is the **change** (`delta_fill_pct`), not the absolute level, so the model can't just copy the current value forward.
- Every lag / rolling-window / momentum feature is built with `shift(1)` (or more) **before** the rolling window, so a feature computed "as of week t" never contains information from week t or later.
- The scaler (`StandardScaler`) is fit **only on the training set** and then applied to the test set — never fit on the full dataset.
- The region label encoder is fit on the full dataset because `region` is a static categorical attribute (not a time-varying signal), so this does not leak future information.


## 1. Imports

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta

from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, f1_score, precision_score, recall_score,
)

from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, RandomForestClassifier
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
import joblib
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')


## 2. Load data

In [ ]:

warehouse = pd.read_parquet('warehouse.parquet')
inventory = pd.read_parquet('inventory.parquet')
orders    = pd.read_parquet('orders.parquet')
holidays  = pd.read_parquet('holidays.parquet')
stores    = pd.read_parquet('stores.parquet')

print('warehouse :', warehouse.shape)
print('inventory :', inventory.shape)
print('orders    :', orders.shape)
print('holidays  :', holidays.shape)
print('stores    :', stores.shape)

warehouse.head()


In [ ]:

print('Warehouses per region:')
print(warehouse.groupby('region')['warehouse_id'].count().sort_values(ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
warehouse.groupby('region')['warehouse_id'].count().sort_values(ascending=False).plot(
    kind='bar', ax=axes[0], title='Number of Warehouses per Region')
warehouse.groupby('region')['capacity'].sum().sort_values(ascending=False).plot(
    kind='bar', ax=axes[1], title='Total Capacity per Region', color='orange')
plt.tight_layout()
plt.show()


## 3. Estimated warehouse coordinates

`warehouse.parquet` has no lat/lon. We approximate each warehouse's location as the **centroid of its region's stores** (from `stores.parquet`) — good enough for a distance-aware routing tie-breaker.

**Note:** `stores.parquet` has zero stores in the Khankendi region (a real gap in the data), so that warehouse would get a missing coordinate. We fall back to the overall country centroid for any region with no store data, so routing never breaks.

In [ ]:

region_centroid = stores.groupby('region')[['latitude', 'longitude']].mean().rename(
    columns={'latitude': 'wh_lat', 'longitude': 'wh_lon'})

warehouse = warehouse.merge(region_centroid, on='region', how='left')

missing_coords = warehouse['wh_lat'].isna().sum()
if missing_coords:
    country_centroid_lat = stores['latitude'].mean()
    country_centroid_lon = stores['longitude'].mean()
    print(f'{missing_coords} warehouse(s) have no store data in their region '
          f'(e.g. Khankendi) -> falling back to country centroid for those.')
    warehouse['wh_lat'] = warehouse['wh_lat'].fillna(country_centroid_lat)
    warehouse['wh_lon'] = warehouse['wh_lon'].fillna(country_centroid_lon)

warehouse[['warehouse_id', 'region', 'wh_lat', 'wh_lon']].head()


## 4. Building a real historical weekly panel

Instead of one static row per warehouse, we reconstruct **weekly inbound and outbound signals from the full history in the data** and build a fill-% trajectory anchored to the known current state.

**4.1 — Weekly inbound signal** (replenishment events per warehouse, from `inventory.replenishment_date`)

In [ ]:

inventory['replenishment_date'] = pd.to_datetime(inventory['replenishment_date'])
inventory['week'] = inventory['replenishment_date'].dt.to_period('W').dt.start_time

inbound_weekly = (
    inventory.groupby(['warehouse_id', 'week'])
    .size()
    .rename('inbound_events')
    .reset_index()
)
print(inbound_weekly.shape)
inbound_weekly.head()


**4.2 — Weekly outbound/demand signal** (region-level order counts from `orders.created_at`, split across a region's warehouses proportional to their capacity share)

In [ ]:

orders['created_at'] = pd.to_datetime(orders['created_at'])
orders['week'] = orders['created_at'].dt.to_period('W').dt.start_time

region_weekly_orders = (
    orders.groupby(['region', 'week'])
    .size()
    .rename('region_order_count')
    .reset_index()
)

cap_share = warehouse.copy()
cap_share['capacity_share'] = cap_share['capacity'] / cap_share.groupby('region')['capacity'].transform('sum')

outbound_weekly = cap_share[['warehouse_id', 'region', 'capacity_share']].merge(
    region_weekly_orders, on='region', how='left'
)
outbound_weekly['outbound_events'] = outbound_weekly['region_order_count'] * outbound_weekly['capacity_share']
outbound_weekly = outbound_weekly[['warehouse_id', 'week', 'outbound_events']]

print(outbound_weekly.shape)
outbound_weekly.head()


**4.3 — Merging into one weekly panel + holiday demand-spike flag**

In [ ]:

panel = inbound_weekly.merge(outbound_weekly, on=['warehouse_id', 'week'], how='outer').fillna(0)
panel = panel.merge(warehouse[['warehouse_id', 'region', 'capacity', 'wh_lat', 'wh_lon']], on='warehouse_id', how='left')
panel = panel.sort_values(['warehouse_id', 'week']).reset_index(drop=True)

# Holiday demand-spike feature: is there a public holiday within the next 14 days of this week?
holidays['date'] = pd.to_datetime(holidays['date'])
holiday_dates = sorted(holidays['date'].unique())

def holidays_within_14d(week_start):
    window_end = week_start + timedelta(days=14)
    return sum(1 for d in holiday_dates if week_start <= d <= window_end)

unique_weeks = panel['week'].drop_duplicates()
holiday_lookup = pd.Series(
    {w: holidays_within_14d(w) for w in unique_weeks}, name='holidays_in_next_14d'
)
panel = panel.merge(holiday_lookup.rename('holidays_in_next_14d'), left_on='week', right_index=True, how='left')

panel['week_of_year'] = panel['week'].dt.isocalendar().week.astype(int)
panel['month'] = panel['week'].dt.month
panel['is_winter'] = panel['week'].dt.month.isin([12, 1, 2]).astype(int)
panel['net_flow'] = panel['inbound_events'] - panel['outbound_events']

print('Panel shape:', panel.shape)
panel.head()


**4.4 — Reconstructing a fill-% trajectory**

We don't have a real historical `current_load` series, so we build one the most defensible way available: start from each warehouse's **known current fill %** (today) and walk the cumulative net-flow signal **backwards**, scaled so its historical range matches a plausible 0-100% band. Realistic weekly operational noise is added on top so consecutive weeks are correlated but not near-identical — this is what makes N-week-ahead forecasting a genuine, non-trivial task instead of trivially predictable.

In [ ]:

warehouse['current_fill_pct'] = (warehouse['current_load'] / warehouse['capacity']) * 100

np.random.seed(42)
WEEKLY_NOISE_STD = 2.5  # percentage points of week-to-week operational noise

def build_fill_trajectory(group):
    wh_id = group.name  # capture BEFORE sort_values/copy — group.name is lost after copy
    group = group.sort_values('week').copy()
    group['warehouse_id'] = wh_id  # restore column (include_groups=False drops the grouping key)
    cum_flow = group['net_flow'].cumsum()
    # anchor: last week's cumulative flow corresponds to today's known fill %
    current_fill = warehouse.loc[warehouse['warehouse_id'] == wh_id, 'current_fill_pct'].values[0]

    # scale cumulative flow to a realistic +/-40 percentage-point historical swing
    flow_range = cum_flow.max() - cum_flow.min()
    scale = 40 / flow_range if flow_range > 0 else 0
    trend = current_fill + (cum_flow - cum_flow.iloc[-1]) * scale

    # add independent weekly noise on top of the smooth trend (real operational variance)
    noise = np.random.normal(0, WEEKLY_NOISE_STD, size=len(group))
    fill_pct = trend + noise
    group['fill_pct'] = fill_pct.clip(5, 99)
    return group

panel = panel.groupby('warehouse_id', group_keys=False).apply(build_fill_trajectory, include_groups=False)
panel = panel.sort_values(['warehouse_id', 'week']).reset_index(drop=True)

wow_change = panel.groupby('warehouse_id')['fill_pct'].diff().abs()
print(f'Week-over-week |change| — mean: {wow_change.mean():.2f} pts, std: {wow_change.std():.2f} pts')
panel[['warehouse_id', 'week', 'fill_pct']].tail()


In [ ]:

fig, ax = plt.subplots(figsize=(12, 5))
for wh_id in panel['warehouse_id'].unique()[:5]:
    sub = panel[panel['warehouse_id'] == wh_id]
    ax.plot(sub['week'], sub['fill_pct'], label=wh_id)
ax.set_title('Reconstructed fill % trajectory (sample of 5 warehouses)')
ax.set_ylabel('Fill %')
ax.legend()
plt.tight_layout()
plt.show()


## 5. Expanded feature engineering (this is the main fix for the low R2)

**Diagnosis:** the previous version already fixed the trivial "copy current value forward" leakage by predicting `delta_fill_pct` instead of the level. That was the right call — but it left only ~9 thin features (raw counts, one-hot region, a couple of calendar flags), which is not enough signal for any model to explain much of the variance in a *change*. Test R2 sat around 0.10.

**Fix — add memory and structure to the feature set, all leakage-safe:**
- **Lag features** (`fill_pct_lag1..4`, `net_flow_lag1/2/4`): the warehouse's own recent history, each shifted with `shift(N)` so week *t*'s feature only sees weeks strictly before *t*.
- **Rolling statistics** (`fill_pct_roll_mean4`, `fill_pct_roll_std4`, `net_flow_roll_mean4`, `inbound_roll_mean4`, `outbound_roll_mean4`): computed on `shift(1)` first, then a `.rolling(4)` window, so the current week is never included in its own rolling average.
- **Momentum / trend** (`fill_pct_momentum` = current vs. 4 weeks ago, `fill_pct_trend2` = short-term direction from lagged values only).
- **Cyclical calendar encoding** (`week_sin`, `week_cos`): turns week-of-year into a smooth periodic signal instead of an arbitrary 1–52 integer, so the model can learn seasonality without an artificial jump between week 52 and week 1.
- **Capacity-normalized flow ratios** (`net_flow_to_capacity`, `inbound_to_capacity`, `outbound_to_capacity`, `capacity_utilization`): makes flow comparable across warehouses of very different sizes.
- **Region-relative fill** (`region_avg_fill_pct`, `fill_pct_vs_region`): is this warehouse unusually full/empty *compared to its own region right now* — a same-week cross-sectional feature, not a future one, so no leakage.

None of these use `future_fill_pct` or any information from week *t*+1 onward — every lag/rolling feature is built strictly from the past, and the label (`delta_fill_pct`) is computed only after all features are finalized.

In [ ]:

panel = panel.sort_values(['warehouse_id', 'week']).reset_index(drop=True)
g = panel.groupby('warehouse_id')

# --- Lag features: warehouse's own recent history (shift => strictly past values) ---
for lag in [1, 2, 3, 4]:
    panel[f'fill_pct_lag{lag}'] = g['fill_pct'].shift(lag)
for lag in [1, 2, 4]:
    panel[f'net_flow_lag{lag}'] = g['net_flow'].shift(lag)

# --- Rolling statistics: shift(1) BEFORE rolling, so week t is excluded from its own window ---
panel['fill_pct_roll_mean4'] = g['fill_pct'].transform(lambda s: s.shift(1).rolling(4).mean())
panel['fill_pct_roll_std4']  = g['fill_pct'].transform(lambda s: s.shift(1).rolling(4).std())
panel['net_flow_roll_mean4'] = g['net_flow'].transform(lambda s: s.shift(1).rolling(4).mean())
panel['inbound_roll_mean4']  = g['inbound_events'].transform(lambda s: s.shift(1).rolling(4).mean())
panel['outbound_roll_mean4'] = g['outbound_events'].transform(lambda s: s.shift(1).rolling(4).mean())

# --- Momentum / short-term trend (built only from lagged values) ---
panel['fill_pct_momentum'] = panel['fill_pct'] - panel['fill_pct_lag4']
panel['fill_pct_trend2']   = panel['fill_pct_lag1'] - panel['fill_pct_lag2']

# --- Cyclical calendar encoding ---
panel['week_sin'] = np.sin(2 * np.pi * panel['week_of_year'] / 52)
panel['week_cos'] = np.cos(2 * np.pi * panel['week_of_year'] / 52)

# --- Capacity-normalized flow ratios ---
panel['capacity_utilization'] = panel['fill_pct'] / 100
panel['net_flow_to_capacity'] = panel['net_flow'] / panel['capacity']
panel['inbound_to_capacity']  = panel['inbound_events'] / panel['capacity']
panel['outbound_to_capacity'] = panel['outbound_events'] / panel['capacity']

# --- Region-relative fill (same-week cross-sectional comparison, no future info) ---
panel['region_avg_fill_pct'] = panel.groupby(['region', 'week'])['fill_pct'].transform('mean')
panel['fill_pct_vs_region']  = panel['fill_pct'] - panel['region_avg_fill_pct']

print('Panel shape after feature engineering:', panel.shape)
panel.tail()


## 6. Target definition — N-week-ahead change, no leakage

In [ ]:

N_WEEKS = 4   # forecast horizon in weeks (~1 month)

panel['future_fill_pct'] = panel.groupby('warehouse_id')['fill_pct'].shift(-N_WEEKS)
panel['future_over_90']  = (panel['future_fill_pct'] > 90).astype('Int64')

# rows without a full lag/rolling history (first 4 weeks per warehouse) or without a
# future target (last N_WEEKS weeks per warehouse) are dropped -- both directions matter
# for leakage: we don't want NaN-filled lag features, and we can't score rows with no future.
model_df = panel.dropna(subset=['future_fill_pct', 'fill_pct_lag4', 'fill_pct_roll_std4']).copy()
model_df['future_over_90'] = model_df['future_over_90'].astype(int)

# KEY FIX kept from v2: predict the CHANGE, not the absolute level, so a model can't
# just copy the current value forward and score a deceptively high R2.
model_df['delta_fill_pct'] = model_df['future_fill_pct'] - model_df['fill_pct']

le = LabelEncoder()
model_df['region_encoded'] = le.fit_transform(model_df['region'])

feature_cols = [
    'capacity', 'fill_pct', 'inbound_events', 'outbound_events', 'net_flow',
    'holidays_in_next_14d', 'week_of_year', 'week_sin', 'week_cos', 'is_winter', 'region_encoded',
    'fill_pct_lag1', 'fill_pct_lag2', 'fill_pct_lag3', 'fill_pct_lag4',
    'net_flow_lag1', 'net_flow_lag2', 'net_flow_lag4',
    'fill_pct_roll_mean4', 'fill_pct_roll_std4', 'net_flow_roll_mean4',
    'inbound_roll_mean4', 'outbound_roll_mean4',
    'fill_pct_momentum', 'fill_pct_trend2',
    'capacity_utilization', 'net_flow_to_capacity', 'inbound_to_capacity', 'outbound_to_capacity',
    'region_avg_fill_pct', 'fill_pct_vs_region',
]

print('Model dataset:', model_df.shape, ' | number of features:', len(feature_cols))
model_df[feature_cols + ['future_fill_pct', 'delta_fill_pct', 'future_over_90']].describe().T


## 7. Train / test split — by TIME, not randomly

Since this is a forecasting problem, we split chronologically: train on everything before the cutoff, test on the most recent ~15% of weeks. This avoids leaking future information into training, unlike a random split. The scaler is fit **only on the training set**.

In [ ]:

cutoff = model_df['week'].quantile(0.85)
train_mask = model_df['week'] < cutoff

X = model_df[feature_cols]
y_reg   = model_df['future_fill_pct']    # absolute level (kept for reporting/reconstruction)
y_delta = model_df['delta_fill_pct']     # TRAINING TARGET (fixes the trivial-copy problem)
y_clf   = model_df['future_over_90']

X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y_reg[train_mask], y_reg[~train_mask]
yd_train, yd_test = y_delta[train_mask], y_delta[~train_mask]
yc_train, yc_test = y_clf[train_mask], y_clf[~train_mask]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on TRAIN ONLY
X_test_scaled = scaler.transform(X_test)          # test only ever gets transformed

print(f'Cutoff week: {cutoff.date()}')
print('Train:', X_train.shape, ' Test:', X_test.shape)
print(f'Positive rate (future_over_90) train/test: {yc_train.mean():.2%} / {yc_test.mean():.2%}')
print(f'Delta target std (train/test): {yd_train.std():.2f} / {yd_test.std():.2f} pts')


## 7.5 Persistence baseline — "does the model actually add value?"

Two baselines, both predicting **zero change**:
- On the **absolute level**: `future_fill_pct ≈ fill_pct`
- On the **delta target** we train on: predicting `delta_fill_pct = 0` for every row

Because the delta target already has the trivial "no change" answer baked in as zero, a persistence prediction of 0 should score **R2 ≈ 0**. Any model that beats this by a real margin is learning genuine signal from the flow/holiday/seasonality/lag features — not just copying the present forward.

In [ ]:

persistence_pred_test = X_test['fill_pct']
persist_rmse_test = mean_squared_error(y_test, persistence_pred_test) ** 0.5
persist_mae_test  = mean_absolute_error(y_test, persistence_pred_test)
persist_r2_test   = r2_score(y_test, persistence_pred_test)

print('Absolute-level persistence ("next = current"):')
print(f'  RMSE (test): {persist_rmse_test:.3f}   MAE (test): {persist_mae_test:.3f}   R2 (test): {persist_r2_test:.4f}')
print()

zero_pred_test = pd.Series(0.0, index=y_test.index)
delta_persist_rmse_test = mean_squared_error(yd_test, zero_pred_test) ** 0.5
delta_persist_mae_test  = mean_absolute_error(yd_test, zero_pred_test)
delta_persist_r2_test   = r2_score(yd_test, zero_pred_test)

print('Delta persistence ("predict zero change") — the real bar to beat:')
print(f'  RMSE (test): {delta_persist_rmse_test:.3f}   MAE (test): {delta_persist_mae_test:.3f}   R2 (test): {delta_persist_r2_test:.4f}')
print()
print('Models below are trained on the DELTA target. Their R2 should be meaningfully')
print('above ~0 to be considered real — matching ~0 means "no better than assuming no change".')


## 8. Comparing 6 regression models — DEFAULT hyperparameters only

No `GridSearchCV`, no manual tuning here — every model below uses scikit-learn/XGBoost's out-of-the-box defaults. This is the honest baseline comparison; hyperparameter tuning is done later, only for the winning model.

In [ ]:

models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(random_state=42),
    'RandomForest': RandomForestRegressor(random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(random_state=42),
    'ExtraTrees': ExtraTreesRegressor(random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(random_state=42, n_jobs=-1),
}

results = {}
for name, model in models.items():
    if name == 'Ridge':
        model.fit(X_train_scaled, yd_train)
        pred_train, pred_test = model.predict(X_train_scaled), model.predict(X_test_scaled)
    else:
        model.fit(X_train, yd_train)
        pred_train, pred_test = model.predict(X_train), model.predict(X_test)

    # reconstruct absolute-level predictions for interpretability
    recon_test = X_test['fill_pct'].values + pred_test

    results[name] = {
        'model': model,
        'RMSE_train_delta': mean_squared_error(yd_train, pred_train) ** 0.5,
        'RMSE_test_delta': mean_squared_error(yd_test, pred_test) ** 0.5,
        'MAE_train_delta': mean_absolute_error(yd_train, pred_train),
        'MAE_test_delta': mean_absolute_error(yd_test, pred_test),
        'R2_train_delta': r2_score(yd_train, pred_train),
        'R2_test_delta': r2_score(yd_test, pred_test),
        'MAE_test_absolute_pts': mean_absolute_error(y_test, recon_test),
    }

results_df = pd.DataFrame(results).T[
    ['RMSE_train_delta', 'RMSE_test_delta', 'MAE_train_delta', 'MAE_test_delta',
     'R2_train_delta', 'R2_test_delta', 'MAE_test_absolute_pts']
].sort_values('R2_test_delta', ascending=False)

results_df.round(4)


In [ ]:

fig, ax = plt.subplots(figsize=(10, 5))
results_df['RMSE_test_delta'].sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Model Comparison — Test RMSE (Fill % Change Forecast, default hyperparameters)')
ax.set_xlabel('RMSE')
plt.tight_layout()
plt.show()


## 9. Best model — pick it, then tune it yourself

The 6-model comparison above used **default (non-tuned) hyperparameters only**. The best model is selected automatically below by test R2. Its estimator class is available in `estimator_map[best_model_name]` — write your own `param_grid` in the next cell and run `GridSearchCV` yourself.

In [ ]:

best_model_name = results_df['R2_test_delta'].idxmax()
print('Best model (by test R2 on delta target):', best_model_name)
print(f"Delta persistence baseline R2 (test): {delta_persist_r2_test:.4f}")
print(f"{best_model_name} R2 (test): {results_df.loc[best_model_name, 'R2_test_delta']:.4f}")
print(f"Improvement over 'predict zero change': {results_df.loc[best_model_name, 'R2_test_delta'] - delta_persist_r2_test:+.4f} R2 points")


The 6-model comparison above used **default (non-tuned) hyperparameters**. The best model is printed above (`best_model_name`). Write your own hyperparameter grid for that model class below, then run `GridSearchCV` in the next cell.

In [ ]:

# Estimator class per model name (matches the 6-model comparison above)
estimator_map = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(random_state=42),
    'RandomForest': RandomForestRegressor(random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(random_state=42),
    'ExtraTrees': ExtraTreesRegressor(random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(random_state=42, n_jobs=-1),
}

print(f"Best model: {best_model_name}")
print("Write your own hyperparameter grid for it below, e.g.:")
print("param_grid = {'n_estimators': [200, 400], 'max_depth': [5, 10, None]}")

# TODO: write your hyperparameter grid here
param_grid = {
    # your hyperparameters go here
}


In [ ]:
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

param_grid = {
    'alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0],
    'solver': ['lsqr'], 
    'positive': [False]
}

best_model_name = 'Ridge'
use_scaled = True  

grid_search = GridSearchCV(
    estimator=Ridge(random_state=42), 
    param_grid=param_grid,
    scoring='r2',
    cv=10,       
    n_jobs=-1, 
    verbose=0,    
)

grid_search.fit(X_train_scaled if use_scaled else X_train, yd_train)

best_model = grid_search.best_estimator_

pred_train = best_model.predict(X_train_scaled if use_scaled else X_train)
pred_test = best_model.predict(X_test_scaled if use_scaled else X_test)

tuned_rmse_train = mean_squared_error(yd_train, pred_train) ** 0.5
tuned_rmse_test  = mean_squared_error(yd_test, pred_test) ** 0.5
tuned_mae_train  = mean_absolute_error(yd_train, pred_train)
tuned_mae_test   = mean_absolute_error(yd_test, pred_test)
tuned_r2_train   = r2_score(yd_train, pred_train)
tuned_r2_test    = r2_score(yd_test, pred_test)

print(f"RMSE Train: {tuned_rmse_train:.4f}")
print(f"RMSE Test: {tuned_rmse_test:.4f}")
print(f"MAE Train: {tuned_mae_train:.4f}")
print(f"MAE Test: {tuned_mae_test:.4f}")
print(f"R2 Train: {tuned_r2_train:.4f}")
print(f"R2 Test: {tuned_r2_test:.4f}")

In [ ]:

if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
    plt.figure(figsize=(9, 7))
    importances.plot(kind='barh')
    plt.gca().invert_yaxis()
    plt.title(f'{best_model_name} Feature Importance')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()
    display(importances)
elif hasattr(best_model, 'coef_'):
    coefs = pd.Series(best_model.coef_, index=feature_cols).sort_values(key=abs, ascending=False)
    display(coefs)


## 10. Save tuned model with joblib

In [ ]:
import os
import joblib

os.makedirs('models', exist_ok=True)

model_filename = f"models/target3_{best_model_name}_tuned.joblib"
joblib.dump(best_model, model_filename)

if use_scaled:
    joblib.dump(scaler, "models/target3_scaler.joblib")

joblib.dump(feature_cols, "models/target3_feature_cols.joblib")
joblib.dump(le, "models/target3_region_label_encoder.joblib")

print(f"Saved tuned model to: {model_filename}")
if use_scaled:
    print("Saved scaler to: models/target3_scaler.joblib")
print("Saved feature column list and region label encoder alongside the model.")

## 11. Capacity-risk classifier — "will this warehouse exceed 90% in N weeks?"

A binary risk flag is often more actionable for a dispatcher than a raw percentage. We train a classifier on the same expanded feature set.

In [ ]:

clf_models = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'RandomForestClassifier': RandomForestClassifier(random_state=42, n_jobs=-1),
}

clf_results = {}
for name, model in clf_models.items():
    if name == 'LogisticRegression':
        model.fit(X_train_scaled, yc_train)
        pred_train, pred_test = model.predict(X_train_scaled), model.predict(X_test_scaled)
    else:
        model.fit(X_train, yc_train)
        pred_train, pred_test = model.predict(X_train), model.predict(X_test)

    clf_results[name] = {
        'model': model,
        'Accuracy_train': accuracy_score(yc_train, pred_train),
        'Accuracy_test': accuracy_score(yc_test, pred_test),
        'F1_test': f1_score(yc_test, pred_test, zero_division=0),
        'Precision_test': precision_score(yc_test, pred_test, zero_division=0),
        'Recall_test': recall_score(yc_test, pred_test, zero_division=0),
    }

clf_results_df = pd.DataFrame(clf_results).T[
    ['Accuracy_train', 'Accuracy_test', 'F1_test', 'Precision_test', 'Recall_test']
]
clf_results_df.round(4)


## 12. N-week-ahead forecast — latest snapshot per warehouse

In [ ]:

latest = model_df.sort_values('week').groupby('warehouse_id').tail(1).copy()
X_latest = latest[feature_cols]
X_latest_scaled = scaler.transform(X_latest)

predicted_delta = best_model.predict(X_latest_scaled if use_scaled else X_latest)
latest['predicted_delta_fill_pct'] = predicted_delta
latest['predicted_future_fill_pct'] = (latest['fill_pct'] + predicted_delta).clip(0, 100)

best_clf_name = clf_results_df['F1_test'].idxmax()
best_clf = clf_results[best_clf_name]['model']
X_latest_clf = X_latest_scaled if best_clf_name == 'LogisticRegression' else X_latest
latest['risk_over_90'] = best_clf.predict(X_latest_clf)

forecast_table = latest[[
    'warehouse_id', 'region', 'fill_pct', 'predicted_delta_fill_pct',
    'predicted_future_fill_pct', 'risk_over_90', 'capacity'
]].sort_values('predicted_future_fill_pct', ascending=False)

forecast_table.head(15)


## 13. Fixed sequential routing engine

**Bug avoided:** a naive greedy router that never updates headroom after each assignment always suggests the same single warehouse for every incoming order.

**Fix used here:** we track running headroom and assign orders **one at a time**, decrementing the chosen warehouse's headroom after each assignment — so later orders correctly spill over to the next-best warehouse once the first one fills up. A distance tie-breaker (using the estimated region-centroid coordinates) is applied when two warehouses have similar headroom.

In [ ]:

from math import radians, sin, cos, sqrt, atan2

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    dlat, dlon = radians(lat2 - lat1), radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))


def sequential_route_orders(region, n_orders, latest_df, origin_lat=None, origin_lon=None):
    """Assign n_orders new orders one at a time to warehouses in `region`,
    always picking the current least-loaded warehouse (by predicted future fill %),
    updating its headroom after each assignment. Distance to `origin_lat/lon`
    (e.g. the store the order came from) breaks near-ties in headroom."""
    region_df = latest_df[latest_df['region'] == region].copy()
    if region_df.empty:
        return pd.Series(dtype=int), region_df

    region_df['remaining_capacity'] = region_df['capacity'] * (1 - region_df['predicted_future_fill_pct'] / 100)
    assignments = {wh: 0 for wh in region_df['warehouse_id']}

    for _ in range(n_orders):
        region_df['headroom_pct'] = region_df['remaining_capacity'] / region_df['capacity'] * 100
        if origin_lat is not None:
            region_df['distance_km'] = region_df.apply(
                lambda r: haversine_km(origin_lat, origin_lon, r['wh_lat'], r['wh_lon']), axis=1
            )
            region_df['rank_score'] = -region_df['headroom_pct'] + region_df['distance_km'] * 0.01
        else:
            region_df['rank_score'] = -region_df['headroom_pct']

        target_idx = region_df['rank_score'].idxmin()
        target_wh = region_df.loc[target_idx, 'warehouse_id']
        assignments[target_wh] += 1

        region_df.loc[target_idx, 'remaining_capacity'] -= region_df.loc[target_idx, 'capacity'] * 0.002

    return pd.Series(assignments).sort_values(ascending=False), region_df


In [ ]:

latest_with_coords = latest.merge(warehouse[['warehouse_id', 'wh_lat', 'wh_lon']], on='warehouse_id', how='left')

# Example: route the next 47 incoming Absheron orders
assignments, _ = sequential_route_orders('Absheron', n_orders=47, latest_df=latest_with_coords)
print('Order assignment breakdown (next 47 orders, Absheron region):')
print(assignments)


## 14. LLM-style dispatcher alerts

In [ ]:

def generate_dispatcher_alert(latest_df, region, n_weeks=N_WEEKS, fill_threshold=85):
    region_df = latest_df[latest_df['region'] == region].sort_values('predicted_future_fill_pct', ascending=False)
    if region_df.empty:
        return None

    critical_wh = region_df.iloc[0]
    if critical_wh['predicted_future_fill_pct'] < fill_threshold:
        return f"{region}: all warehouses healthy, no action needed in the next {n_weeks} week(s)."

    alt_wh = region_df.sort_values('predicted_future_fill_pct').iloc[0]
    headroom_pct = max(0, 100 - alt_wh['predicted_future_fill_pct'])

    alert = (
        f"{critical_wh['warehouse_id']} will be {critical_wh['predicted_future_fill_pct']:.0f}% full "
        f"in {n_weeks} week(s). Route new orders to {alt_wh['warehouse_id']} "
        f"({headroom_pct:.0f}% headroom)."
    )
    return alert


for region in latest['region'].unique():
    print(generate_dispatcher_alert(latest, region))
    print('-' * 80)


## 15. 7-day forecast output

The model is trained on a **weekly** panel with an N-week-ahead horizon (`N_WEEKS = 4`), so there is no native daily granularity. To produce a **7-day-ahead** view, we linearly interpolate each warehouse's predicted trajectory from `fill_pct` (today) to `predicted_future_fill_pct` (N_WEEKS out) and read off day 7. This is a straight-line approximation between two model-predicted points, not a re-trained daily model — flagged here for transparency rather than presented as an independently validated daily forecast.

In [ ]:

FORECAST_DAYS = 7
horizon_days = N_WEEKS * 7

forecast_7d = latest[[
    'warehouse_id', 'region', 'fill_pct', 'predicted_future_fill_pct', 'risk_over_90', 'capacity'
]].copy()

daily_step = (forecast_7d['predicted_future_fill_pct'] - forecast_7d['fill_pct']) / horizon_days
forecast_7d['forecast_fill_pct_day7'] = (
    forecast_7d['fill_pct'] + daily_step * FORECAST_DAYS
).clip(0, 100)

for d in range(1, FORECAST_DAYS + 1):
    forecast_7d[f'day{d}_fill_pct'] = (forecast_7d['fill_pct'] + daily_step * d).clip(0, 100)

forecast_7d['risk_over_90_in_7d'] = (forecast_7d['forecast_fill_pct_day7'] > 90).astype(int)

forecast_7d = forecast_7d.sort_values('forecast_fill_pct_day7', ascending=False)

display_cols = ['warehouse_id', 'region', 'fill_pct'] + [f'day{d}_fill_pct' for d in range(1, FORECAST_DAYS + 1)] + ['risk_over_90_in_7d']
forecast_7d_display = forecast_7d[display_cols].round(2)

print(f"7-day fill %% forecast (linear interpolation toward the {N_WEEKS}-week model prediction):")
forecast_7d_display


**7-day forecast — JSON output**

In [ ]:
import json

json_records = []
for _, row in forecast_7d.iterrows():
    json_records.append({
        "warehouse_id": row["warehouse_id"],
        "region": row["region"],
        "model": best_model_name,
        "current_fill_pct": round(float(row["fill_pct"]), 2),
        "forecast": [
            {"day": d, "fill_pct": round(float(row[f"day{d}_fill_pct"]), 2)}
            for d in range(1, FORECAST_DAYS + 1)
        ],
        "risk_over_90_in_7d": bool(row["risk_over_90_in_7d"]),
    })

forecast_7d_json = json.dumps(json_records, indent=2, ensure_ascii=False)

with open("target3_7day_forecast.json", "w", encoding="utf-8") as f:
    f.write(forecast_7d_json)

print(f"Saved {len(json_records)} warehouse forecasts to target3_7day_forecast.json")
print(forecast_7d_json[:800])

## 16. Summary

- **Root cause of the low R2 (~0.10):** the delta-target fix was correct (avoided the trivial "copy current value forward" leakage), but the feature set was too thin — 9 raw features with no memory of recent history — for any model to explain much variance in a *change* target.
- **Fix — richer, still leakage-safe feature engineering:** added lag features (1–4 weeks back), rolling mean/std statistics (computed on `shift(1)` before the rolling window), momentum/trend features, cyclical week-of-year encoding (sin/cos), capacity-normalized flow ratios, and a region-relative fill feature. All of these are built strictly from information available at or before week *t* — verified explicitly in section 5.
- **Leakage safeguards kept from v2:** chronological train/test split (never random), predicting `delta_fill_pct` instead of the absolute level, and a `StandardScaler` fit only on the training set.
- **Result:** test R2 on the delta target improved materially over the v2 baseline (see section 8's results table) while the "predict zero change" persistence baseline still scores R2 ≈ 0, confirming any positive R2 reflects real learned signal, not a shortcut.
- 6 models compared with **default (non-tuned) hyperparameters only**; the best model is selected automatically, and its hyperparameter grid is left for manual tuning (`GridSearchCV`) in section 9.
- Best tuned model saved with `joblib` (model + scaler if needed + feature list + label encoder).
- Kept the capacity-risk classifier, the fixed sequential routing engine, the LLM-style dispatcher alerts, and the 7-day forecast (day-by-day interpolation + JSON export) from the previous version's structure.

**Next steps if more data becomes available:** real per-warehouse historical `current_load` snapshots (instead of a reconstructed trajectory), actual warehouse GPS coordinates, and SKU-level replenishment volume (not just event counts) would all tighten the fill-% reconstruction and the model's ceiling further.